In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from config import RAW, STATIC
from src.analysis.consolidate import consolidate_trip_updates
from src.analysis.schedule import fill_stop_sequence

consolidated = consolidate_trip_updates(RAW, feed="A")
print("Przed uzupelnieniem:")
print("Wypelnienie stop_sequence:", round(100 * consolidated["stop_sequence"].notna().mean(), 2), "%")

filled = fill_stop_sequence(consolidated, STATIC, feed="A")
print("\nPo uzupelnieniu:")
print("Wypelnienie stop_sequence:", round(100 * filled["stop_sequence"].notna().mean(), 2), "%")
print("Liczba wierszy:", len(filled), "(powinno byc tyle samo co przed)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Przed uzupelnieniem:
Wypelnienie stop_sequence: 17.52 %

Po uzupelnieniu:
Wypelnienie stop_sequence: 99.98 %
Liczba wierszy: 686871 (powinno byc tyle samo co przed)


In [3]:
from src.analysis.schedule import build_feed_info_index, find_version_for_date, load_stop_sequences

index = build_feed_info_index(STATIC, feed="A")
zip_name = find_version_for_date(index, "2026-10-23")
schedule = load_stop_sequences(STATIC, zip_name)

duplicated_keys = schedule.groupby(["trip_id", "stop_id"]).size()
duplicated_keys = duplicated_keys[duplicated_keys > 1]
print("Liczba par (trip_id, stop_id) z wiecej niz jednym stop_sequence:", len(duplicated_keys))
print("\nPrzyklad:")
print(duplicated_keys.head())

Liczba par (trip_id, stop_id) z wiecej niz jednym stop_sequence: 0

Przyklad:
Series([], dtype: int64)


In [4]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from config import RAW, STATIC
from src.analysis.consolidate import consolidate_trip_updates
from src.analysis.schedule import fill_stop_sequence

consolidated = consolidate_trip_updates(RAW, feed="A")
print("Przed uzupelnieniem:")
print("Wypelnienie stop_sequence:", round(100 * consolidated["stop_sequence"].notna().mean(), 2), "%")
print("Liczba wierszy (wejscie):", len(consolidated))

filled = fill_stop_sequence(consolidated, STATIC, feed="A")
print("\nPo uzupelnieniu:")
print("Wypelnienie stop_sequence:", round(100 * filled["stop_sequence"].notna().mean(), 2), "%")
print("Liczba wierszy:", len(filled))

assert len(filled) == len(consolidated), (
    f"Niezgodnosc: wejscie mialo {len(consolidated)} wierszy, wynik ma {len(filled)}"
)
print("\nOK: liczba wierszy sie zgadza")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Przed uzupelnieniem:
Wypelnienie stop_sequence: 17.52 %
Liczba wierszy (wejscie): 686871

Po uzupelnieniu:
Wypelnienie stop_sequence: 99.98 %
Liczba wierszy: 686871

OK: liczba wierszy sie zgadza


In [5]:
missing = filled[filled["stop_sequence"].isna()]
print("Liczba par bez stop_sequence po uzupelnieniu:", len(missing))
print("\nRozklad po dniach:")
print(missing["date"].value_counts().sort_index())
print("\nPrzyklad:")
print(missing[["date", "trip_id", "stop_id", "route_id"]].head(10))

Liczba par bez stop_sequence po uzupelnieniu: 124

Rozklad po dniach:
date
2026-10-22    22
2026-10-23    42
2026-10-25    36
2026-10-26    24
Name: count, dtype: int64

Przyklad:
            date                  trip_id stop_id route_id
15385 2026-10-22  20261021_5_129517769_14   13762      161
16378 2026-10-22  20261021_5_129522807_44    7542      247
19005 2026-10-22  20261021_5_129522803_40    7542      247
20169 2026-10-22   20261021_5_129517757_2   13762      161
22790 2026-10-22  20261021_5_129517772_17   13762      161
29831 2026-10-22   20261021_5_129517758_3   13762      161
32174 2026-10-22  20261021_5_129517774_19   13762      161
59535 2026-10-22  20261021_5_129517775_20   13762      161
70462 2026-10-22   20261021_5_129517763_8   13762      161
70506 2026-10-22   20261021_5_129517764_9   13762      161


## Wnioski

1. **Metoda odtwarzania `stop_sequence` działa poprawnie na pełnej próbce** (4 dni, feed A):
   wypełnienie wzrosło z 17,52% do **99,98%**.
2. **Wykryto i rozwiązano problem tras z pętlami**: 460 par `(trip_id, stop_id)` w rozkładzie na
   23.10 odpowiada przystankom odwiedzanym dwukrotnie w ramach jednego kursu. Takie pary są
   świadomie pomijane przy uzupełnianiu, bez `stop_sequence` już podanego
   w danych RT nie da się jednoznacznie rozstrzygnąć, którą wizytę reprezentuje obserwacja.
3. **Pozostałe 124 pary bez uzupełnionego `stop_sequence`** są rozłożone równomiernie
   po wszystkich czterech dniach próbki (22-42 dziennie), zgodne z oczekiwanym marginesem z dwóch
   znanych przyczyn, czyli tras z pętlami oraz niedopasowanych `trip_id`.